In [165]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os

In [166]:
paths = ["raw//"+ x for x in os.listdir("raw") if x.endswith('.csv')]

In [167]:
import os

print(os.getcwd())
for p in paths:
    print(p, "->", os.path.exists(p))


c:\Users\mohak\OneDrive\Documents\DATASET\NAD_retry
raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv -> True
raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv -> True
raw//Friday-WorkingHours-Morning.pcap_ISCX.csv -> True
raw//Monday-WorkingHours.pcap_ISCX.csv -> True
raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> True
raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv -> True
raw//Tuesday-WorkingHours.pcap_ISCX.csv -> True
raw//Wednesday-workingHours.pcap_ISCX.csv -> True


In [168]:
paths


['raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'raw//Monday-WorkingHours.pcap_ISCX.csv',
 'raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'raw//Tuesday-WorkingHours.pcap_ISCX.csv',
 'raw//Wednesday-workingHours.pcap_ISCX.csv']

In [169]:
test_path = paths[:3]
train_path = ['raw//Monday-WorkingHours.pcap_ISCX.csv',
             'raw//Tuesday-WorkingHours.pcap_ISCX.csv',
             'raw//Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
             'raw//Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
             'raw//Wednesday-workingHours.pcap_ISCX.csv']
test_path = test_path[::-1]
test_path

['raw//Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'raw//Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']

In [170]:
columns_list = [pd.read_csv(p,nrows=0).columns.to_list() for p in paths]
  
all_same = all(cols == columns_list[0] for cols in columns_list)
print("All datasets have matching columns:", all_same)

All datasets have matching columns: True


In [171]:
dfs=[pd.read_csv(p) for p in train_path]

merged_df_train = pd.concat(dfs,axis=0,ignore_index=True) 


dfs=[pd.read_csv(p) for p in test_path]

merged_df_test = pd.concat(dfs,axis=0,ignore_index=True) 
print(len(merged_df_train)+len(merged_df_test))

#total should be 2830743 rows

2830743


In [172]:
merged_df_test.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [173]:
# merged_df_test = merged_df_test.drop([' Destination Port'],axis=1)
# merged_df_train = merged_df_train.drop([' Destination Port'],axis=1)

In [174]:
merged_df_train[' Label'].value_counts()

 Label
BENIGN                        1858775
DoS Hulk                       231073
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Combining Exteremely Small Classes into 1

In [175]:
SMALL_ATTACKS = [
    'Infiltration',
    'Heartbleed'
]

NEW_LABEL = "Other Attacks"
merged_df_train[' Label'] = merged_df_train[' Label'].replace(SMALL_ATTACKS, NEW_LABEL)

Fixing encoding Issues

In [176]:
import re

def clean_web_attack_labels(df):
    
    # 1. Clean 'Web Attack - Brute Force'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*Brute Force$', 
        'Web Attack - Brute Force', 
        regex=True
    )

    # 2. Clean 'Web Attack - XSS'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*XSS$', 
        'Web Attack - XSS', 
        regex=True
    )
    
    # 3. Clean 'Web Attack - Sql Injection'
    df[' Label'] = df[' Label'].str.replace(
        r'^Web Attack.*Sql Injection$', 
        'Web Attack - Sql Injection', 
        regex=True
    )
    
    return df

merged_df_train = clean_web_attack_labels(merged_df_train.copy())

Removing Whitespace in the front

In [177]:
col_names_train = {col: col.strip() for col in merged_df_train.columns}
merged_df_train.rename(columns = col_names_train, inplace = True)
col_names_test = {col: col.strip() for col in merged_df_test.columns}
merged_df_test.rename(columns = col_names_test, inplace = True)

In [178]:
merged_df_train['Label'].value_counts()

Label
BENIGN                        1858775
DoS Hulk                       231073
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Other Attacks                      47
Web Attack - Sql Injection         21
Name: count, dtype: int64

In [179]:
merged_df_test['Label'].value_counts()

Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64

Mapping labels to Attacks

In [180]:
attack_type_train = {
    'BENIGN': 'BENIGN',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'DoS Hulk': 'DoS',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Web Attack - Brute Force': 'Web Attack',
    'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
    'Other Attacks': 'Other Attacks'
}

attack_type_test={
    'DDoS': 'DDoS',
    'PortScan': 'Port Scan',
    'BENIGN': 'BENIGN',
    'Bot': 'Bot',
}
merged_df_test['Attack'] = merged_df_test['Label'].map(attack_type_test)
merged_df_train['Attack'] = merged_df_train['Label'].map(attack_type_train)

In [181]:
merged_df_train['Attack'].value_counts()

Attack
BENIGN           1858775
DoS               252661
Brute Force        13835
Web Attack          2180
Other Attacks         47
Name: count, dtype: int64

In [182]:
merged_df_test['Attack'].value_counts()

Attack
BENIGN       414322
Port Scan    158930
DDoS         128027
Bot            1966
Name: count, dtype: int64

Ensuring all attacks are covered in Train set

In [183]:
bot_df = merged_df_test[merged_df_test["Attack"]=="Bot"]
ddos_df = merged_df_test[merged_df_test["Attack"]=="DDoS"]
port_df = merged_df_test[merged_df_test["Attack"]=="Port Scan"]
benign_df = merged_df_test[merged_df_test["Attack"]=="BENIGN"]

BOT_TRAIN_FRAC = 0.50
RANDOM_SEED = 42

bot_train = bot_df.sample(frac=BOT_TRAIN_FRAC,random_state=RANDOM_SEED)
port_train = port_df.sample(n=30000,random_state=RANDOM_SEED)
ddos_train = ddos_df.sample(n=30000,random_state=RANDOM_SEED)

bot_test = bot_df.drop(bot_train.index)
port_test = port_df.drop(port_train.index)
ddos_test = ddos_df.drop(ddos_train.index)

merged_df_test = pd.concat([bot_test,port_test,ddos_test,benign_df],axis=0)
merged_df_train = pd.concat([merged_df_train,bot_train,port_train,ddos_train],axis=0)


In [184]:
merged_df_train['Attack'].value_counts()

Attack
BENIGN           1858775
DoS               252661
Port Scan          30000
DDoS               30000
Brute Force        13835
Web Attack          2180
Bot                  983
Other Attacks         47
Name: count, dtype: int64

In [185]:
merged_df_test['Attack'].value_counts()

Attack
BENIGN       414322
Port Scan    128930
DDoS          98027
Bot             983
Name: count, dtype: int64

Adding missing attacks to test set

In [ ]:
RANDOM_SEED = 42

# individual class subsets
dos_df       = merged_df_train[merged_df_train["Attack"]=="DoS"]
brute_df        = merged_df_train[merged_df_train["Attack"]=="Brute Force"]
web_df      = merged_df_train[merged_df_train["Attack"]=="Web Attack"]
other_df      = merged_df_train[merged_df_train["Attack"]=="Other Attacks"]

benign_df     = merged_df_train[merged_df_train["Attack"]=="BENIGN"]
bot_df = merged_df_train[merged_df_train["Attack"]=="Bot"]
ddos_df = merged_df_train[merged_df_train["Attack"]=="DDoS"]
port_df = merged_df_train[merged_df_train["Attack"]=="Port Scan"]


# samples for test (for missing classes)
dos_test      = dos_df.sample(n=6000, random_state=RANDOM_SEED)
brute_test       = brute_df.sample(frac=0.2, random_state=RANDOM_SEED)
web_test       = web_df.sample(frac=0.2, random_state=RANDOM_SEED)
other_test     = other_df.sample(frac=0.5, random_state=RANDOM_SEED)

# remaining go back to train
dos_train      = dos_df.drop(dos_test.index)
brute_train     = brute_df.drop(brute_test.index)
web_train       = web_df.drop(web_test.index)
other_train     = other_df.drop(other_test.index)




# new test additions
new_test_parts = [
    dos_test,
    web_test,
    brute_test,
    other_test
]

new_test_df = pd.concat(new_test_parts, axis=0)

# remove these samples from old train
new_train_parts = [
    dos_train,
    web_train,
    brute_train,
    other_train,
    benign_df,
    bot_df,
    port_df,
    ddos_df
]

merged_df_train = pd.concat(new_train_parts, axis=0)
merged_df_test  = pd.concat([merged_df_test, new_test_df], axis=0)

In [187]:
merged_df_train['Attack'].value_counts()

Attack
BENIGN           1858775
DoS               246661
Port Scan          30000
DDoS               30000
Brute Force        11068
Web Attack          1744
Bot                  983
Other Attacks         23
Name: count, dtype: int64

In [188]:
merged_df_test['Attack'].value_counts()

Attack
BENIGN           414322
Port Scan        128930
DDoS              98027
DoS                6000
Brute Force        2767
Bot                 983
Web Attack          436
Other Attacks        24
Name: count, dtype: int64

In [189]:
merged_df_test.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

In [190]:
merged_df_train.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

Removing Duplicates

In [191]:
dups_train = merged_df_train[merged_df_train.duplicated()]
print(f"Number of Train Duplicates: {len(dups_train)}")
dups_test = merged_df_test[merged_df_test.duplicated()]
print(f"Number of Test Duplicates: {len(dups_test)}")

Number of Train Duplicates: 203521
Number of Test Duplicates: 70723


In [192]:
print(f"Train shape before dropping Duplicates: {merged_df_train.shape}")
merged_df_train.drop_duplicates(inplace=True)
print(f"Train shape after dropping Duplicates: {merged_df_train.shape}")
print(f"Test shape before dropping Duplicates: {merged_df_test.shape}")
merged_df_test.drop_duplicates(inplace=True)
print(f"Test shape after dropping Duplicates: {merged_df_test.shape}")

Train shape before dropping Duplicates: (2179254, 80)
Train shape after dropping Duplicates: (1975733, 80)
Test shape before dropping Duplicates: (651489, 80)
Test shape after dropping Duplicates: (580766, 80)


In [193]:
merged_df_train['Attack'].value_counts()

Attack
BENIGN           1720966
DoS               189135
DDoS               29999
Port Scan          25501
Brute Force         7417
Web Attack          1718
Bot                  974
Other Attacks         23
Name: count, dtype: int64

In [194]:
merged_df_test['Attack'].value_counts()

Attack
BENIGN           395106
DDoS              98022
Port Scan         79290
DoS                4871
Brute Force        2039
Bot                 981
Web Attack          433
Other Attacks        24
Name: count, dtype: int64

Handling missing and infinity

In [195]:
missing_val_train = merged_df_train.isna().sum()
print("Train:")
print(missing_val_train.loc[missing_val_train > 0])

missing_val_test = merged_df_test.isna().sum()
print("Test:")
print(missing_val_test.loc[missing_val_test > 0])

Train:
Flow Bytes/s    307
dtype: int64
Test:
Flow Bytes/s    50
dtype: int64


In [196]:
# Checking for infinity values
numeric_cols = merged_df_train.select_dtypes(include = np.number).columns
inf_count = np.isinf(merged_df_train[numeric_cols]).sum()
print("Train Inf Values")
print(inf_count[inf_count > 0])

numeric_cols = merged_df_test.select_dtypes(include = np.number).columns
inf_count = np.isinf(merged_df_test[numeric_cols]).sum()
print("Test Inf Values")
print(inf_count[inf_count > 0])

Train Inf Values
Flow Bytes/s       834
Flow Packets/s    1141
dtype: int64
Test Inf Values
Flow Bytes/s      393
Flow Packets/s    443
dtype: int64


In [197]:
# Replacing any infinite values (positive or negative) with NaN (not a number)
print(f'Initial missing values Train: {merged_df_train.isna().sum().sum()}')

merged_df_train.replace([np.inf, -np.inf], np.nan, inplace = True)

print(f'Missing values after processing infinite values for Train: {merged_df_train.isna().sum().sum()}')

print(f'Initial missing values Train: {merged_df_test.isna().sum().sum()}')

merged_df_test.replace([np.inf, -np.inf], np.nan, inplace = True)

print(f'Missing values after processing infinite values for Train: {merged_df_test.isna().sum().sum()}')
     

Initial missing values Train: 307
Missing values after processing infinite values for Train: 2282
Initial missing values Train: 50
Missing values after processing infinite values for Train: 886


In [198]:

missing_train = merged_df_train.isna().sum()
print("Train:")
print(missing_train.loc[missing_train > 0])

missing_test = merged_df_test.isna().sum()
print("Test:")
print(missing_test.loc[missing_test > 0])

Train:
Flow Bytes/s      1141
Flow Packets/s    1141
dtype: int64
Test:
Flow Bytes/s      443
Flow Packets/s    443
dtype: int64


In [199]:

# Calculating missing value percentage in the dataset
mis_per = (missing_train / len(merged_df_train)) * 100
mis_table = pd.concat([missing_train, mis_per.round(2)], axis = 1)
print("Train:")
mis_table = mis_table.rename(columns = {0 : 'Missing Values', 1 : 'Percentage of Total Values'})
print(mis_table.loc[mis_per > 0])

mis_per = (missing_test / len(merged_df_test)) * 100
mis_table = pd.concat([missing_test, mis_per.round(2)], axis = 1)
print("\n\nTest:")
mis_table = mis_table.rename(columns = {0 : 'Missing Values', 1 : 'Percentage of Total Values'})
print(mis_table.loc[mis_per > 0])

Train:
                Missing Values  Percentage of Total Values
Flow Bytes/s              1141                        0.06
Flow Packets/s            1141                        0.06


Test:
                Missing Values  Percentage of Total Values
Flow Bytes/s               443                        0.08
Flow Packets/s             443                        0.08


In [200]:
median_flow_bytes_train = merged_df_train['Flow Bytes/s'].median()
median_flow_packets_train = merged_df_train['Flow Packets/s'].median()

print('Train Median of Flow Bytes/s: ', median_flow_bytes_train)
print('Train Median of Flow Packets/s: ', median_flow_packets_train)

median_flow_bytes_test = merged_df_test['Flow Bytes/s'].median()
median_flow_packets_test = merged_df_test['Flow Packets/s'].median()

print('Train Median of Flow Bytes/s: ', median_flow_bytes_test)
print('Train Median of Flow Packets/s: ', median_flow_packets_test)



Train Median of Flow Bytes/s:  3364.4174645000003
Train Median of Flow Packets/s:  76.66653889
Train Median of Flow Bytes/s:  6644.518272
Train Median of Flow Packets/s:  77.93321124


In [201]:
merged_df_train['Flow Bytes/s'].fillna(median_flow_bytes_train, inplace = True)
merged_df_train['Flow Packets/s'].fillna(median_flow_packets_train, inplace = True)
merged_df_test['Flow Bytes/s'].fillna(median_flow_bytes_test, inplace = True)
merged_df_test['Flow Packets/s'].fillna(median_flow_packets_test, inplace = True)

In [202]:
print('Number of \'Flow Bytes/s\' missing values:', merged_df_train['Flow Bytes/s'].isna().sum())
print('Number of \'Flow Packets/s\' missing values:', merged_df_train['Flow Packets/s'].isna().sum())
print('Number of \'Flow Bytes/s\' missing values:', merged_df_test['Flow Bytes/s'].isna().sum())
print('Number of \'Flow Packets/s\' missing values:', merged_df_test['Flow Packets/s'].isna().sum())

Number of 'Flow Bytes/s' missing values: 0
Number of 'Flow Packets/s' missing values: 0
Number of 'Flow Bytes/s' missing values: 0
Number of 'Flow Packets/s' missing values: 0


In [203]:
merged_df_train['Attack'].value_counts()

Attack
BENIGN           1720966
DoS               189135
DDoS               29999
Port Scan          25501
Brute Force         7417
Web Attack          1718
Bot                  974
Other Attacks         23
Name: count, dtype: int64

In [204]:
merged_df_test['Attack'].value_counts()

Attack
BENIGN           395106
DDoS              98022
Port Scan         79290
DoS                4871
Brute Force        2039
Bot                 981
Web Attack          433
Other Attacks        24
Name: count, dtype: int64

In [205]:
merged_df_train.drop('Label', axis = 1, inplace = True)
merged_df_test.drop('Label', axis = 1, inplace = True)

In [206]:
merged_df_test.shape

(580766, 79)

In [207]:
# TARGET_BENIGN_COUNT = 30000

# benign_mask = (merged_df_train[' Label']=="BENIGN")


# benign_train_undersampled = merged_df_train[benign_mask].sample(
#     n = TARGET_BENIGN_COUNT,
#     random_state=RANDOM_SEED
# )
# print(f"Sampled BENIGN count: {len(benign_train_undersampled):,}")

# attack_mask = (merged_df_train[' Label'] != 'BENIGN')
# attack_train_df = merged_df_train[attack_mask]
# print(f"Total Attack count preserved: {len(attack_train_df):,}")


# train_df_final = pd.concat(
#     [attack_train_df,benign_train_undersampled],
#     axis=0
# )

In [208]:
# #for test df
# TARGET_BENIGN_COUNT = 6000

# benign_mask = (test_df_final[' Label']=="BENIGN")


# benign_train_undersampled = test_df_final[benign_mask].sample(
#     n = TARGET_BENIGN_COUNT,
#     random_state=RANDOM_SEED
# )
# print(f"Sampled BENIGN count: {len(benign_train_undersampled):,}")

# attack_mask = (test_df_final[' Label'] != 'BENIGN')
# attack_train_df = test_df_final[attack_mask]
# print(f"Total Attack count preserved: {len(attack_train_df):,}")


# test_df_final = pd.concat(
#     [attack_train_df,benign_train_undersampled],
#     axis=0
# )

In [209]:
# #for test df
# TARGET_DDOS_COUNT = 6000

# ddos_mask = (test_df_final[' Label']=="DDoS")


# ddos_train_undersampled = test_df_final[ddos_mask].sample(
#     n = TARGET_DDOS_COUNT,
#     random_state=RANDOM_SEED
# )
# print(f"Sampled DDoS count: {len(ddos_train_undersampled):,}")

# attack_mask = (test_df_final[' Label'] != 'DDoS')
# attack_train_df = test_df_final[attack_mask]
# print(f"Total Attack count preserved: {len(attack_train_df):,}")


# test_df_final = pd.concat(
#     [attack_train_df,ddos_train_undersampled],
#     axis=0
# )

In [210]:
# #for test df
# TARGET_PORTSCAN_COUNT = 6000

# portscan_mask = (test_df_final[' Label']=="PortScan")


# portscan_train_undersampled = test_df_final[portscan_mask].sample(
#     n = TARGET_PORTSCAN_COUNT,
#     random_state=RANDOM_SEED
# )
# print(f"Sampled PortScan count: {len(portscan_train_undersampled):,}")

# attack_mask = (test_df_final[' Label'] != 'PortScan')
# attack_train_df = test_df_final[attack_mask]
# print(f"Total Attack count preserved: {len(attack_train_df):,}")


# test_df_final = pd.concat(
#     [attack_train_df,portscan_train_undersampled],
#     axis=0
# )

In [211]:
# TARGET_DOS_COUNT = 30000

# dos_mask = (train_df_final[' Label']=="DoS Hulk")


# dos_train_undersampled = train_df_final[dos_mask].sample(
#     n = TARGET_DOS_COUNT,
#     random_state=RANDOM_SEED
# )
# print(f"Sampled Dos Hulk count: {len(dos_train_undersampled):,}")

# attack_mask = (train_df_final[' Label'] != 'DoS Hulk')
# attack_train_df = train_df_final[attack_mask]
# print(f"Total Attack count preserved: {len(attack_train_df):,}")


# train_df_final = pd.concat(
#     [attack_train_df,dos_train_undersampled],
#     axis=0
# )

In [213]:
merged_df_train.to_parquet('train.parquet')
merged_df_test.to_parquet('test.parquet')